# DAKGEA - Data Augmentation with PLM

In this notebook we'll add **data augmentation** to the previous experiment.

Augmentation uses a **Pre-trained Language Model (PLM)** to generate synthetic entities that improve alignment.

## Setup

In [ ]:
import sys
from pathlib import Path
import yaml
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from experiments.runner.runner import ExperimentRunner
from src.config.loader import load_yaml

## Configuration with Augmentation

Compared to the previous notebook, we add the **augmentation** section.

In [ ]:
config = {
    "experiment": {
        "name": "tutorial_augmented",
        
        # Dataset
        "dataset": {
            "name": "openea/D_W_15K_V1",
            "writer": "bert_int"
        },
        
        # Reduction: 30% of entities
        "reduction": {
            "method": "random_entities",
            "ratio": 0.3,
            "writer": "bert_int",
            "eval": True,
            "save_dataset": False,
            "save_model": False
        },
        
        # NEW SECTION: Augmentation
        "augmentation": {
            "method": "plm",                # Use PLM (BART)
            "ratio": 0.5,                   # Generate 50% more entities
            "writer": {
                "type": "bert_int",
                "augmented_only_train": True  # Synthetic entities only in training
            },
            "eval": True,                   # Also evaluate with augmentation
            "save_dataset": False,
            "save_model": False
        },
        
        "model": "bert_int",
        "seed": 42,
        "clear": True,
        "overwrite_existing": False
    }
}

print(yaml.dump(config, default_flow_style=False))

## How Augmentation Works

1. **Reduction**: Select 30% of original entities
2. **Augmentation**: 
   - Analyze reduced entities
   - Use BART (PLM) to generate similar synthetic entities
   - Generate 50% more data (ratio=0.5)
3. **Training**: 
   - Training set: original + synthetic entities
   - Test set: original entities only
4. **Evaluation**: Evaluate on real entities

In [ ]:
# Save configuration
config_file = PROJECT_ROOT / "config" / "experiments" / "tutorial_augmented.yaml"

with open(config_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✓ Config saved to: {config_file}")

## Run Experiment

⚠️ **Note**: Augmentation takes longer than baseline (10-20 minutes)

In [ ]:
# Load and create runner
config_data = load_yaml(config_file)
runner = ExperimentRunner(
    payload=config_data,
    cli_overwrite=False,
    default_overwrite=False
)

print(f"Experiment: {runner.name}")
print(f"Stages: {len(runner.augmentations) > 0 and 'reduction + augmentation' or 'reduction only'}")

In [ ]:
# RUN
print("Starting augmented experiment...")
print("This will take 10-20 minutes.\n")

try:
    runner.run()
    print("\n✓ Experiment completed!")
except Exception as e:
    print(f"\n✗ Failed: {e}")
    import traceback
    traceback.print_exc()

## Comparison: Baseline vs Augmented

In [ ]:
# Load baseline results (from previous notebook)
baseline_file = PROJECT_ROOT / "results" / "tutorial_baseline" / "reduction_030" / "D_W_15K_V1" / "reduction" / "results.json"

# Load augmented results
augmented_file = runner.base_workspace / "reduction_030" / "D_W_15K_V1" / "augmentation" / "results.json"

def load_results(file_path):
    if file_path.exists():
        with open(file_path) as f:
            return json.load(f)
    return None

baseline_results = load_results(baseline_file)
augmented_results = load_results(augmented_file)

if baseline_results and augmented_results:
    print("=" * 80)
    print("COMPARISON: Baseline vs Augmented")
    print("=" * 80)
    
    model = "bert_int"
    baseline = baseline_results.get(model, {})
    augmented = augmented_results.get(model, {})
    
    metrics = ["hits@1", "hits@5", "hits@10", "mrr", "precision", "recall", "f-measure"]
    
    print(f"\n{'Metric':<15} {'Baseline':<12} {'Augmented':<12} {'Δ':<12} {'Improvement'}")
    print("-" * 80)
    
    for metric in metrics:
        base_val = baseline.get(metric, 0)
        aug_val = augmented.get(metric, 0)
        delta = aug_val - base_val
        improvement = (delta / base_val * 100) if base_val > 0 else 0
        
        symbol = "✓" if delta > 0 else "✗" if delta < 0 else "="
        
        print(f"{metric:<15} {base_val:<12.4f} {aug_val:<12.4f} {delta:+.4f}      {symbol} {improvement:+.2f}%")
    
    print("\n" + "=" * 80)
    print("Legend:")
    print("  ✓ = Improvement")
    print("  ✗ = Degradation")
    print("  = = No change")
else:
    if not baseline_results:
        print("⚠ Baseline results not found. Run notebook 02 first.")
    if not augmented_results:
        print("⚠ Augmented results not found. Check if experiment completed successfully.")

## Interpreting Results

### What to Expect

- **Hits@1 improves**: Augmentation increases top-1 precision
- **Recall improves**: More synthetic data helps find more alignments
- **Improvement varies**: Depends on dataset and parameters

### Tuning Parameters

- **reduction.ratio**: How many original entities to use (0.1 - 1.0)
- **augmentation.ratio**: How many synthetic entities to generate (0.1 - 1.0)
- **augmentation.method**: Which method to use (plm, stub, etc.)

### Common Combinations

```yaml
# Low data, high augmentation
reduction.ratio: 0.1    # 10% original data
augmentation.ratio: 1.0  # 100% synthetic data

# Balanced data
reduction.ratio: 0.3    # 30% original data
augmentation.ratio: 0.5  # 50% synthetic data

# High data, low augmentation
reduction.ratio: 0.8    # 80% original data
augmentation.ratio: 0.2  # 20% synthetic data
```

## Detailed Analysis of Generated Data

We can inspect the generated synthetic entities:

In [ ]:
# Load augmentation summary
summary_file = runner.base_workspace / "reduction_030" / "D_W_15K_V1" / "augmentation" / "summary.json"

if summary_file.exists():
    with open(summary_file) as f:
        summary = json.load(f)
    
    print("Augmentation Summary:")
    print("-" * 40)
    print(f"Method: {summary.get('method', 'N/A')}")
    print(f"Ratio: {summary.get('ratio', 'N/A')}")
    print(f"Original pairs: {summary.get('original_pairs', 'N/A')}")
    print(f"Synthetic pairs: {summary.get('synthetic_pairs', 'N/A')}")
    print(f"Total pairs: {summary.get('total_pairs', 'N/A')}")
else:
    print("Summary file not found")

## Next Steps

In the next notebook we'll see:
- How to analyze results in detail
- How to generate plots and statistics
- How to compare multiple configurations

➡️ Continue with: **04_analyze_results.ipynb**